In [1]:
import json
import time
import pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

密钥读取

In [2]:
secret_client = UserSecretsClient()
api_key = secret_client.get_secret("ZHIPU_API_KEY")
client = OpenAI(
    api_key=api_key,
    base_url="https://open.bigmodel.cn/api/paas/v4/"
)

环境原始数据（文档原样复制）

In [3]:
environment_data = [
    {"date":"11-01 周六","weather":"多云","temp_high":29,"temp_low":21,"wind":"东北风1级","event":"无","passenger_coeff":1.3,"random_factor":1.02},
    {"date":"11-02 周日","weather":"多云","temp_high":29,"temp_low":22,"wind":"东北风2级","event":"无","passenger_coeff":1.3,"random_factor":0.97},
    {"date":"11-03 周一","weather":"阴/小雨","temp_high":25,"temp_low":20,"wind":"北风2级","event":"无","passenger_coeff":1.0,"random_factor":0.92},
    {"date":"11-04 周二","weather":"多云","temp_high":24,"temp_low":19,"wind":"北风1级","event":"无","passenger_coeff":1.0,"random_factor":1.05},
    {"date":"11-05 周三","weather":"多云","temp_high":31,"temp_low":21,"wind":"北风2级","event":"无","passenger_coeff":1.0,"random_factor":1.08},
    {"date":"11-06 周四","weather":"晴","temp_high":31,"temp_low":21,"wind":"北风2级","event":"无","passenger_coeff":1.0,"random_factor":1.03},
    {"date":"11-07 周五","weather":"晴","temp_high":32,"temp_low":22,"wind":"东北风1级","event":"潜水展Day1","passenger_coeff":1.4,"random_factor":1.06},
    {"date":"11-08 周六","weather":"多云/晴","temp_high":33,"temp_low":22,"wind":"西南风1级","event":"潜水展Day2","passenger_coeff":1.5,"random_factor":1.11},
    {"date":"11-09 周日","weather":"晴/多云","temp_high":33,"temp_low":23,"wind":"东北风2级","event":"潜水展Day3","passenger_coeff":1.5,"random_factor":0.95},
    {"date":"11-10 周一","weather":"阴/小雨","temp_high":29,"temp_low":23,"wind":"北风3级","event":"无","passenger_coeff":1.0,"random_factor":0.93},
    {"date":"11-11 周二","weather":"小雨/阴","temp_high":26,"temp_low":22,"wind":"北风2级","event":"金鸡百花电影节开幕","passenger_coeff":1.6,"random_factor":1.07},
    {"date":"11-12 周三","weather":"多云/阴","temp_high":24,"temp_low":19,"wind":"西北风2级","event":"金鸡电影节","passenger_coeff":1.6,"random_factor":1.04},
    {"date":"11-13 周四","weather":"多云","temp_high":26,"temp_low":21,"wind":"北风2级","event":"金鸡电影节","passenger_coeff":1.6,"random_factor":0.98},
    {"date":"11-14 周五","weather":"晴","temp_high":27,"temp_low":18,"wind":"北风2级","event":"金鸡电影节","passenger_coeff":1.7,"random_factor":1.09},
    {"date":"11-15 周六","weather":"晴/多云","temp_high":27,"temp_low":17,"wind":"北风2级","event":"金鸡电影节闭幕","passenger_coeff":1.7,"random_factor":1.01},
    {"date":"11-16 周日","weather":"晴/中雨","temp_high":29,"temp_low":18,"wind":"北风1级","event":"无","passenger_coeff":1.2,"random_factor":0.88},
    {"date":"11-17 周一","weather":"阴","temp_high":27,"temp_low":18,"wind":"东北风2级","event":"无","passenger_coeff":1.0,"random_factor":1.06},
    {"date":"11-18 周二","weather":"多云/晴","temp_high":22,"temp_low":14,"wind":"北风2级","event":"无","passenger_coeff":1.0,"random_factor":0.96},
    {"date":"11-19 周三","weather":"晴","temp_high":21,"temp_low":14,"wind":"东北风2级","event":"无","passenger_coeff":1.0,"random_factor":1.03},
    {"date":"11-20 周四","weather":"多云","temp_high":25,"temp_low":14,"wind":"北风2级","event":"咖啡展Day0(布展)","passenger_coeff":1.2,"random_factor":1.05},
    {"date":"11-21 周五","weather":"晴","temp_high":25,"temp_low":14,"wind":"北风2级","event":"咖啡展+ISPO展开幕","passenger_coeff":1.5,"random_factor":1.10},
    {"date":"11-22 周六","weather":"晴","temp_high":25,"temp_low":14,"wind":"西北风2级","event":"咖啡展+ISPO展","passenger_coeff":1.6,"random_factor":1.07},
    {"date":"11-23 周日","weather":"晴","temp_high":28,"temp_low":15,"wind":"北风2级","event":"咖啡展+ISPO展闭幕","passenger_coeff":1.6,"random_factor":0.99},
    {"date":"11-24 周一","weather":"多云","temp_high":27,"temp_low":15,"wind":"北风2级","event":"无","passenger_coeff":1.0,"random_factor":1.02},
    {"date":"11-25 周二","weather":"多云","temp_high":26,"temp_low":16,"wind":"东北风3级","event":"无","passenger_coeff":1.0,"random_factor":0.95},
    {"date":"11-26 周三","weather":"晴","temp_high":20,"temp_low":14,"wind":"东北风4级","event":"无","passenger_coeff":1.0,"random_factor":1.04},
    {"date":"11-27 周四","weather":"阴","temp_high":21,"temp_low":15,"wind":"东北风3级","event":"无","passenger_coeff":1.0,"random_factor":0.97},
    {"date":"11-28 周五","weather":"多云","temp_high":22,"temp_low":11,"wind":"东北风4级","event":"无","passenger_coeff":1.2,"random_factor":1.01},
    {"date":"11-29 周六","weather":"晴/多云","temp_high":24,"temp_low":12,"wind":"东南风3级","event":"无","passenger_coeff":1.3,"random_factor":1.06},
    {"date":"11-30 周日","weather":"晴/多云","temp_high":26,"temp_low":15,"wind":"东南风1级","event":"无","passenger_coeff":1.3,"random_factor":0.94},
]

奶茶店模拟器类

In [4]:
class ShopSimulator:
    def __init__(self):
        self.money = 100000.0
        self.stock = 0
        self.reputation = 50
        self.current_day = 1
        self.is_bankrupt = False
        self.shortage_occur = False
        self.price = 10
        self.ad_budget = 0
        self.ingredient_cost = 4

    def get_state(self):
        return {
            "money": round(self.money,2),
            "stock": self.stock,
            "reputation": self.reputation,
            "current_day": self.current_day
        }

    def get_environment(self):
        return environment_data[self.current_day -1]

    def get_forecast(self):
        day_info = self.get_environment()
        base = 50
        price_factor = max(0.1, 1 - (self.price -10)/20)
        w = day_info["weather"]
        f = 1.0
        if "晴" in w:
            f *=1.0
        elif "阴" in w:
            f *=0.9
        if "小雨" in w:
            f *=0.7
        if "中雨" in w:
            f *=0.65
        high = day_info["temp_high"]
        low = day_info["temp_low"]
        if high >=30:
            f *=1.15
        if low <=15:
            f *=0.85
        weather_factor = f
        passenger_factor = day_info["passenger_coeff"]
        ad_factor = 1 + self.ad_budget /500
        rep_factor = 1 + (self.reputation -50)/100
        rand = day_info["random_factor"]
        demand = base * price_factor * weather_factor * passenger_factor * ad_factor * rep_factor * rand
        return {"forecast_demand": int(round(demand,0))}

    def set_price(self, price: float):
        self.price = max(1, price)
        return {"ok":True, "price_set":self.price}

    def buy_ingredients(self, buy_quantity:int):
        cost = buy_quantity * self.ingredient_cost
        if self.money >= cost:
            self.money -= cost
            self.stock += buy_quantity
            return {"ok":True, "buy_quantity":buy_quantity}
        else:
            return {"ok":False, "msg":"资金不足"}

    def run_ad(self, budget:int):
        if self.money >= budget:
            self.money -= budget
            self.ad_budget = budget
            return {"ok":True, "ad_budget":budget}
        else:
            return {"ok":False, "msg":"广告预算资金不足"}

    def calc_demand(self):
        day_info = self.get_environment()
        base = 50
        price_factor = max(0.1, 1 - (self.price -10)/20)
        w = day_info["weather"]
        f = 1.0
        if "晴" in w:
            f *=1.0
        elif "阴" in w:
            f *=0.9
        if "小雨" in w:
            f *=0.7
        if "中雨" in w:
            f *=0.65
        high = day_info["temp_high"]
        low = day_info["temp_low"]
        if high >=30:
            f *=1.15
        if low <=15:
            f *=0.85
        weather_factor = f
        passenger_factor = day_info["passenger_coeff"]
        ad_factor = 1 + self.ad_budget /500
        rep_factor = 1 + (self.reputation -50)/100
        rand = day_info["random_factor"]
        demand = base * price_factor * weather_factor * passenger_factor * ad_factor * rep_factor * rand
        # 修改：真实需求取整
        return int(round(demand,0))

    def daily_step(self):
        demand = self.calc_demand()
        self.shortage_occur = False
        if self.stock >= demand:
            sell = demand
        else:
            sell = self.stock
            self.shortage_occur = True
        revenue = sell * self.price
        self.money += revenue
        self.stock -= sell
        # 口碑变动
        if self.shortage_occur:
            self.reputation = max(0, self.reputation - 2)
        else:
            self.reputation = min(100, self.reputation + 1)
        # 重置当日广告
        self.ad_budget = 0
        record = {
            "day":self.current_day,
            "money":round(self.money,2),
            "stock":self.stock,
            "price":self.price,
            "demand":demand,
            "sell":sell,
            "reputation":self.reputation,
            "shortage":self.shortage_occur,
            "event":self.get_environment()["event"]
        }
        self.current_day +=1
        if self.money <=0:
            self.is_bankrupt = True
        return record


工具定义

In [5]:
tools = [
    {
        "type":"function",
        "function":{
            "name":"get_state",
            "description":"获取店铺当前状态：资金、库存、口碑、当前天数",
            "parameters":{"type":"object","properties":{},"required":[]}
        }
    },
    {
        "type":"function",
        "function":{
            "name":"get_environment",
            "description":"获取今日环境信息：日期、天气、温度、活动、客流系数",
            "parameters":{"type":"object","properties":{},"required":[]}
        }
    },
    {
        "type":"function",
        "function":{
            "name":"get_forecast",
            "description":"预测今日需求量",
            "parameters":{"type":"object","properties":{},"required":[]}
        }
    },
    {
        "type":"function",
        "function":{
            "name":"set_price",
            "description":"设置奶茶单价",
            "parameters":{"type":"object","properties":{"price":{"type":"number"}},"required":["price"]}
        }
    },
    {
        "type":"function",
        "function":{
            "name":"buy_ingredients",
            "description":"采购原料，参数buy_quantity为采购份数",
            "parameters":{"type":"object","properties":{"buy_quantity":{"type":"integer"}},"required":["buy_quantity"]}
        }
    },
    {
        "type":"function",
        "function":{
            "name":"run_ad",
            "description":"投放广告，参数budget为广告投入金额",
            "parameters":{"type":"object","properties":{"budget":{"type":"integer"}},"required":["budget"]}
        }
    }
]

主循环 agent_loop

In [6]:
def agent_loop(sim:ShopSimulator, client):
    records = []
    sys_prompt = """
你是奶茶店经营智能体，经营流程：
1. 调用get_state获取店铺当前资金、库存、口碑；
2. 调用get_environment获取今日外部环境；
3. 调用get_forecast获取今日需求预测；
4. 根据上面3个信息，决策set_price定价、buy_ingredients采购原料；可选run_ad投放广告。

硬性规则：
1. 不允许只调用查询工具，必须执行set_price、buy_ingredients；
2. 原料成本固定4元/份，定价必须高于成本，活动日提高采购量，防止缺货；
3. 你可以分多轮调用工具，每拿到工具返回结果再做下一步决策；
4. 只输出function call，不要多余文字解释。
"""
    while sim.current_day <=30 and not sim.is_bankrupt:
        print(f"\n----------开始第{sim.current_day}天经营----------")
        messages = [{"role":"system","content":sys_prompt}]
        messages.append({"role":"user","content":"开始今日经营，先查询店铺状态、环境、需求，然后执行定价和采购！"})

        has_set_price = False
        has_buy = False
        max_round = 8
        for _ in range(max_round):
            completion = client.chat.completions.create(
                model="glm-4-flash",
                messages=messages,
                tools=tools,
                tool_choice="auto"
            )
            resp = completion.choices[0].message
            tool_calls = resp.tool_calls
            if not tool_calls:
                break
            # 把assistant消息（对象）加入messages，会自动转成dict
            messages.append(resp.model_dump())
            call_names = [tc.function.name for tc in tool_calls]
            # 执行工具
            for tc in tool_calls:
                fname = tc.function.name
                args = json.loads(tc.function.arguments)
                print(f"工具调用 {fname} → {args}")
                if fname == "get_state":
                    res = sim.get_state()
                elif fname == "get_environment":
                    res = sim.get_environment()
                elif fname == "get_forecast":
                    res = sim.get_forecast()
                elif fname == "set_price":
                    res = sim.set_price(args["price"])
                    has_set_price = True
                elif fname == "buy_ingredients":
                    res = sim.buy_ingredients(args["buy_quantity"])
                    has_buy = True
                elif fname == "run_ad":
                    res = sim.run_ad(args["budget"])
                else:
                    res = {"msg":"未知工具"}
                messages.append({
                    "role":"tool",
                    "tool_call_id":tc.id,
                    "content":json.dumps(res, ensure_ascii=False)
                })
            # 判断是否完成定价+采购
            if has_set_price and has_buy:
                break

        # =========兜底判断，现在直接用标记变量=========
        if not (has_set_price and has_buy):
            print("⚠模型没有生成定价/采购动作，启用兜底经营策略！")
            forecast = sim.get_forecast()
            demand_pred = forecast["forecast_demand"]
            sim.set_price(15)
            sim.buy_ingredients(int(demand_pred + 5))
            sim.run_ad(0)

        # 【修改点】这里用daily_step，匹配你的模拟器！
        day_record = sim.daily_step()
        records.append(day_record)
        print(f">>>第{sim.current_day-1}天结算结果：{day_record}")
        if sim.is_bankrupt:
            print("店铺破产，模拟终止！")
            break
    return records


导出CSV

In [7]:
def export_csv(records, save_path="/kaggle/working/shop_result.csv"):
    df = pd.DataFrame(records)
    # 新增：强制三列为整数
    df["stock"] = df["stock"].astype(int)
    df["demand"] = df["demand"].astype(int)
    df["sell"] = df["sell"].astype(int)
    df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"\nCSV已保存至: {save_path}")
    return df


启动模拟

In [8]:
simulator = ShopSimulator()
records = agent_loop(simulator, client)
df = export_csv(records)
display(df)


----------开始第1天经营----------
工具调用 get_state → {}
工具调用 get_environment → {}
工具调用 get_forecast → {}
工具调用 set_price → {'price': 6}
工具调用 buy_ingredients → {'buy_quantity': 70}
>>>第1天结算结果：{'day': 1, 'money': 100140.0, 'stock': 0, 'price': 6, 'demand': 80, 'sell': 70, 'reputation': 48, 'shortage': True, 'event': '无'}

----------开始第2天经营----------
工具调用 get_state → {}
工具调用 get_environment → {}
工具调用 get_forecast → {}
工具调用 set_price → {'price': 8}
工具调用 buy_ingredients → {'buy_quantity': 100}
>>>第2天结算结果：{'day': 2, 'money': 100284.0, 'stock': 32, 'price': 8, 'demand': 68, 'sell': 68, 'reputation': 49, 'shortage': False, 'event': '无'}

----------开始第3天经营----------
工具调用 get_state → {}
工具调用 get_environment → {}
工具调用 get_forecast → {}
工具调用 set_price → {'price': 6}
工具调用 buy_ingredients → {'buy_quantity': 40}
>>>第3天结算结果：{'day': 3, 'money': 100328.0, 'stock': 38, 'price': 6, 'demand': 34, 'sell': 34, 'reputation': 50, 'shortage': False, 'event': '无'}

----------开始第4天经营----------
工具调用 get_state → {}
工具调用 ge

,day,money,stock,price,demand,sell,reputation,shortage,event
0,1,100140.0,0,6.0,80,70,48,True,无
1,2,100284.0,32,8.0,68,68,49,False,无
2,3,100328.0,38,6.0,34,34,50,False,无
3,4,100512.0,50,8.0,58,58,51,False,无
4,5,100682.0,45,6.0,75,75,52,False,无
5,6,100850.0,69,8.0,66,66,53,False,无
6,7,101146.0,92,8.0,97,97,54,False,潜水展Day1
7,8,101546.0,102,8.0,110,110,55,False,潜水展Day2
8,9,101564.0,149,6.0,103,103,56,False,潜水展Day3
9,10,101688.0,152,8.0,34,34,57,False,无
